# Caso C · 05 Validación supervisada con etiquetas — matriz por tipo

> _Tutorial · Caso de uso: **C — Anomalías HVAC** · Capa Medallion: **oro** · Spec: `docs/specs/synthetic-bms/02-domain-spec.md`_

Material docente del proyecto **CAPTIA Synthetic Data BMS** — IES Dr. Lluís Simarro,
Curso de Especialización IA & Big Data 2025-2026.


## 1. Objetivo

Reportar precision, recall y F1 por tipo de fallo, no solo AUC global.


## 2. Qué se aprende

- Reportes desglosados.
- Trade-off precision vs recall.
- Cómo escoger umbral según coste de FN.


## 3. Contexto del caso de uso

En CENTINELA+ un FN (no detectar) es peor que un FP (alarma falsa). Tunear umbral con eso en mente.


## 4. Relación con CENTINELA+

Validación final antes de servir el modelo.


## 5. Relación con Medallion

Oro.


## 6. Datos de entrada

Mock + scores del notebook anterior.


## 7. Schema CAPTIA esperado

No aplica.


## 8. Setup y variables de entorno

Cargamos las variables de entorno (`.env`), inicializamos `numpy` con `seed=42` y aplicamos el estilo de plotting compartido. Los helpers viven en `notebooks/_common/`.


In [ ]:
# Setup canónico — todos los notebooks didácticos lo usan
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT.name and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from notebooks._common.captia_schema import (
    CANONICAL_TAGS, MEASUREMENT_TELEMETRY, MEASUREMENT_FAULT_LABELS,
    DEFAULT_BUCKET_RETENTIONS, KNOWN_VARIABLES,
    build_topic, build_line_protocol, validate_canonical_tags,
)
from notebooks._common.connection import load_env, get_influx_client, get_default_bucket
from notebooks._common.plotting import setup_default_style, plot_timeseries, plot_distribution
import notebooks._common.synthetic_mocks as mocks

SEED = 42
rng = np.random.default_rng(SEED)
setup_default_style()
load_env()
print(f"ROOT={ROOT}, SEED={SEED}, default_bucket={get_default_bucket()}")


## 9. Carga de datos o mock

Recomputamos rápidamente.


In [ ]:
df, _ = mocks.make_lbnl_fdd_rtu_mock()
X = pd.DataFrame({
    "dt_supply_return": df["RA_TEMP"] - df["SA_TEMP"],
    "valve": df["CCV"], "fan": df["FAN_STATE"],
}).dropna()
y = df["is_fault"].astype(int).iloc[X.index]
ftype = df["fault_type"].iloc[X.index]


## 10. Exploración paso a paso

Train IF rápido (clase).


In [ ]:
from sklearn.ensemble import IsolationForest
iso = IsolationForest(contamination=0.15, random_state=SEED).fit(X)
score = -iso.score_samples(X)


## 11. Transformación bronce → plata

No aplica.


## 12. Construcción de capa oro

Threshold por percentil 90.


In [ ]:
import numpy as np
threshold = np.percentile(score, 90)
y_pred = (score >= threshold).astype(int)
print(f"threshold={threshold:.3f}; predicted positives={y_pred.sum()}")


## 13. Visualizaciones explicativas

Tabla precision/recall por tipo.


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score
rows = []
for t in ftype.unique():
    if t == "normal":
        continue
    mask = (ftype == t) | (ftype == "normal")
    yt = (ftype[mask] == t).astype(int)
    yp = y_pred[mask]
    rows.append({
        "fault": t,
        "precision": precision_score(yt, yp, zero_division=0),
        "recall": recall_score(yt, yp, zero_division=0),
        "f1": f1_score(yt, yp, zero_division=0),
        "support": int(yt.sum()),
    })
report = pd.DataFrame(rows).set_index("fault").round(3)
report


## 14. Validaciones

Recall por tipo no debe ser cero (sería detector ciego).


In [ ]:
assert (report["recall"] > 0.05).all()


## 15. Errores comunes

1. Reportar solo macro F1 — esconde tipos raros.
2. Comparar threshold % sin recalibrar entre runs.
3. Mezclar período de entrenamiento con período de evaluación.


## 16. Ejercicios propuestos

1. Calcula la matriz de confusión cruzada.
2. Aplica `class_weight` en un modelo supervisado RF y compara.
3. Diseña una métrica con coste asimétrico (FN cuesta 5x FP).


## 17. Cómo se reutiliza con datos reales

Cuando llegue un ticket en producción, evaluar precision/recall/F1 sobre la ventana etiquetada. Re-entrenar si recall < 0.7.


## 18. Resumen final y próximos pasos

Recuerda los conceptos principales del notebook y enlaza al siguiente paso.

- Siguiente notebook: `04_case_D_iaq_occupancy/01_eda_iaq_ocupacion.ipynb`.
- Documento web del caso: `docs/validation/ml-validation.md`.
